<a href="https://colab.research.google.com/github/jyizheng/my-study/blob/main/colab/SlidingWindowAttention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import torch
import torch.nn as nn
import math

def sliding_window_attention_mask(sequence_length, window_size):
    """
    创建一个滑动窗口注意力掩码。

    参数:
    - sequence_length (int): 输入序列的长度。
    - window_size (int): 注意力窗口的大小。窗口是双向的，
                         每个 token 关注其左右各 (window_size - 1) / 2 个 token。
                         建议使用奇数以保证对称。

    返回:
    - torch.Tensor: 一个形状为 [sequence_length, sequence_length] 的布尔掩码张量。
                    True 表示该位置被遮盖（不参与注意力计算）。
    """
    if window_size % 2 == 0:
        raise ValueError("Window size should be an odd number for a symmetric window.")

    # 计算单侧窗口大小
    half_window_size = (window_size - 1) // 2

    # 创建一个表示所有位置索引的张量
    # arange(N) -> [0, 1, 2, ..., N-1]
    # .unsqueeze(1) -> [[0], [1], ..., [N-1]] (形状: N x 1)
    # .unsqueeze(0) -> [[0, 1, ..., N-1]] (形状: 1 x N)
    indices = torch.arange(sequence_length).unsqueeze(1)
    indices_right = torch.arange(sequence_length).unsqueeze(0)

    # 计算每个位置与其他位置的相对距离
    # 广播机制会使其成为一个 N x N 的矩阵，值为 j - i
    relative_distance = indices_right - indices
    print(relative_distance)

    # 创建掩码：如果一个位置在窗口之外，则将其标记为 True
    # abs(j - i) > w/2
    mask = torch.abs(relative_distance) > half_window_size

    return mask

class SlidingWindowAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, window_size):
        """
        初始化滑动窗口多头注意力层。

        参数:
        - embed_dim (int): 输入嵌入的维度。
        - num_heads (int): 注意力头的数量。
        - window_size (int): 滑动窗口的大小。
        """
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError("Embedding dimension must be divisible by the number of heads.")

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.window_size = window_size

        # 定义 Q, K, V 的线性变换层
        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x, mask=None):
        """
        前向传播。

        参数:
        - x (torch.Tensor): 输入张量，形状为 [batch_size, sequence_length, embed_dim]。
        - mask (torch.Tensor, optional): 外部传入的额外掩码 (例如 padding mask)。

        返回:
        - torch.Tensor: 注意力层的输出，形状与输入相同。
        """
        batch_size, seq_len, _ = x.shape

        # 1. 生成 Q, K, V
        # qkv: [batch_size, seq_len, embed_dim * 3]
        qkv = self.qkv_proj(x)

        # 将 qkv 分割成 Q, K, V，并为多头注意力重塑形状
        # 最终形状: [batch_size, num_heads, seq_len, head_dim]
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # 2. 计算注意力得分
        # (B, H, N, D) @ (B, H, D, N) -> (B, H, N, N)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 3. 应用滑动窗口掩码
        # sliding_mask 的形状是 [seq_len, seq_len]
        sliding_mask = sliding_window_attention_mask(seq_len, self.window_size).to(x.device)

        # 扩展掩码以匹配得分张量的形状 [B, H, N, N]
        # (B, H, N, N) 中的注意力得分会被加上一个非常大的负数
        scores = scores.masked_fill(sliding_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        # (可选) 应用外部传入的 padding mask
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        # 4. 计算注意力权重并应用于 V
        attn_weights = torch.softmax(scores, dim=-1)

        # (B, H, N, N) @ (B, H, N, D) -> (B, H, N, D)
        context = torch.matmul(attn_weights, v)

        # 5. 合并多头结果
        # [B, H, N, D] -> [B, N, H, D] -> [B, N, H*D]
        context = context.permute(0, 2, 1, 3).contiguous().view(batch_size, seq_len, self.embed_dim)

        # 6. 最终的线性投影
        output = self.out_proj(context)

        return output

# --- 示例用法 ---
if __name__ == '__main__':
    # 参数设置
    batch_size = 4
    sequence_length = 512  # 一个较长的序列
    embed_dim = 128
    num_heads = 8
    window_size = 33     # 窗口大小 (奇数)

    res = sliding_window_attention_mask(15, 3)
    print(res)

    # 创建模型和输入
    attention_layer = SlidingWindowAttention(embed_dim, num_heads, window_size)
    input_tensor = torch.randn(batch_size, sequence_length, embed_dim)

    # 运行模型
    output = attention_layer(input_tensor)

    print(f"输入形状: {input_tensor.shape}")
    print(f"输出形状: {output.shape}")

    # 验证掩码
    mask_example = sliding_window_attention_mask(10, 5)
    print("\n滑动窗口掩码示例 (序列长度=10, 窗口大小=5):")
    # False 表示可以关注，True 表示被遮盖
    print(mask_example)

tensor([[  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
          14],
        [ -1,   0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
          13],
        [ -2,  -1,   0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,
          12],
        [ -3,  -2,  -1,   0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,
          11],
        [ -4,  -3,  -2,  -1,   0,   1,   2,   3,   4,   5,   6,   7,   8,   9,
          10],
        [ -5,  -4,  -3,  -2,  -1,   0,   1,   2,   3,   4,   5,   6,   7,   8,
           9],
        [ -6,  -5,  -4,  -3,  -2,  -1,   0,   1,   2,   3,   4,   5,   6,   7,
           8],
        [ -7,  -6,  -5,  -4,  -3,  -2,  -1,   0,   1,   2,   3,   4,   5,   6,
           7],
        [ -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,   0,   1,   2,   3,   4,   5,
           6],
        [ -9,  -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,   0,   1,   2,   3,   4,
           5],
        [-10,  -9,  -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,  